In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("spark-q6") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/07 20:32:04 WARN Utils: Your hostname, Pratyushs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.29.229 instead (on interface en0)
26/03/07 20:32:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/07 20:32:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-07 20:32:21--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:2179:dc00:b:20a5:b140:21, 2600:9000:2179:400:b:20a5:b140:21, 2600:9000:2179:bc00:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:2179:dc00:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-07 20:32:22 (784 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [5]:
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")

In [6]:
zones = spark.read.option("header", "true") \
    .csv("taxi_zone_lookup.csv")

In [7]:
df_join = df.join(
    zones,
    df.PULocationID == zones.LocationID
)

In [8]:
zone_counts = df_join.groupBy("Zone") \
    .count() \
    .orderBy(col("count"))

In [9]:
zone_counts.show(5)

[Stage 4:=============================>                             (4 + 4) / 8]

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Governor's Island...|    1|
|Eltingville/Annad...|    1|
|       Arden Heights|    1|
|       Port Richmond|    3|
|       Rikers Island|    4|
+--------------------+-----+
only showing top 5 rows
